In [55]:
import os
import numpy as np
import itertools
import random

import graph_tool.all as gt
from graph_tool.spectral import adjacency
from src.gwi.geometric_weights_inference import geometric_weights_inference

In [56]:
from scipy.sparse import csr_matrix


def _sparse_estimator_wrapper(x: csr_matrix, estimator: str, **kwargs):
    return getattr(x, estimator)(**kwargs).A1


def _sparse_sum(x: csr_matrix, **kwargs):
    return _sparse_estimator_wrapper(x, "sum", **kwargs)

In [57]:
base_path = '../dataset/ATLAS'
network = 'covert_FIFA2015'
network_path = network + '.gt'
method_name = 'CND'
top_node = 0.01
add_link = 0.01

In [ ]:
filename = os.path.join(base_path, network_path) 

g = gt.Graph()
g.load(filename)

print("Graph loaded successfully!")
print(f"File: {filename}")
print(f"Number of vertices (nodes): {g.num_vertices()}")
print(f"Number of edges (links): {g.num_edges()}")


network_result = geometric_weights_inference(
    graph=g,
    weightings=[method_name],
)

node_values = _sparse_sum(network_result.weighting_results[method_name], axis=1)

num_nodes = g.num_vertices()

top_k_count = max(1, int(num_nodes * top_node))

sorted_indices = np.argsort(node_values)

top_nodes_indices = sorted_indices[-top_k_count:]

print(f"Identified the top {top_node*100}% of nodes.")
print(f"Total nodes in graph: {num_nodes}")
print(f"Number of top nodes selected: {len(top_nodes_indices)}")
print(f"Indices of the top nodes: \n{top_nodes_indices}")


g_robust = g.copy()
print(f"Created a copy of the graph. Initial edges: {g_robust.num_edges()}")

total_links_added = 0

for node_index in top_nodes_indices:
    target_node = g.vertex(node_index)

    neighbors = list(target_node.out_neighbors())

    if len(neighbors) < 2:
        continue

    missing_links = []
    for u, v in itertools.combinations(neighbors, 2):

        if g_robust.edge(u, v) is None:
            missing_links.append((u, v))
    
    print(f"Node {node_index} has {len(missing_links)} missing links to add with {len(neighbors)} neighbors.")
    
    if not missing_links:
        continue

    num_to_add = max(1, int(len(missing_links) * add_link))
    
    num_to_add = min(num_to_add, len(missing_links))

    links_to_add = random.sample(missing_links, num_to_add)

    for u, v in links_to_add:
        new_edge = g_robust.add_edge(u, v)
        
        if 'weight' in g_robust.ep:
            g_robust.ep.weight[new_edge] = 1.0
    
    total_links_added += len(links_to_add)

print(f"\nFinished processing all top nodes.")
print(f"Total new links added: {total_links_added}")
print(f"Nodes in original graph 'g':      {g.num_vertices()}")
print(f"Edges in original graph 'g':      {g.num_edges()}")
print(f"Nodes in new robust graph 'g_robust': {g_robust.num_vertices()}")
print(f"Edges in new robust graph 'g_robust': {g_robust.num_edges()}")

top_nodes_p_str = int(top_node * 100)
links_added_p_str = int(add_link * 100)

adjacency_matrix = adjacency(g_robust, csr=True).astype(
copy=False, dtype=np.float64
)
if adjacency_matrix.diagonal().sum() != 0:
    raise ValueError("No self-loops are allowed")

if adjacency_matrix.min() < 0 or adjacency_matrix.max() > 1:
    print(adjacency_matrix.min(), adjacency_matrix.max())
    raise ValueError("Network must be unweighted")


new_filename = f"{base_path}/{network}_robust_{method_name}_nodes{str(top_nodes_p_str)}p_links{links_added_p_str}p.gt"

g_robust.save(new_filename)
print(f"\nSuccessfully saved the robust graph to: {new_filename}")

Graph loaded successfully!
File: ../dataset/ATLAS/covert_FIFA2015.gt
Number of vertices (nodes): 414
Number of edges (links): 4805
Computing weighting for current_weighting='CND' took 0.00 s
Identified the top 1.0% of nodes.
Total nodes in graph: 414
Number of top nodes selected: 4
Indices of the top nodes: 
[ 53 157 127  92]
Created a copy of the graph. Initial edges: 4805
Node 53 has 3475 missing links to add with 96 neighbors.
Node 157 has 3490 missing links to add with 96 neighbors.
Node 127 has 4282 missing links to add with 105 neighbors.
Node 92 has 5155 missing links to add with 115 neighbors.

Finished processing all top nodes.
Total new links added: 161
Nodes in original graph 'g':      414
Edges in original graph 'g':      4805
Nodes in new robust graph 'g_robust': 414
Edges in new robust graph 'g_robust': 4966

Successfully saved the robust graph to: ../dataset/ATLAS/covert_FIFA2015_robust_CND_nodes1p_links1p.gt
